## Codypharm – career chatbot with tools and evaluator

This notebook builds my resume bot using the LinkedIn PDF and summary. It uses **tool calling** (record interest, log unknown questions, read/write a SQLite Q&A store), a **TOOL_MAP** for dispatch, and an **evaluator** (critique-and-refine) that can retry the reply once if it fails a quality check.

**What I build:**

- **Tools:** `record_user_details`, `record_unknown_question`, `query_qa`, `upsert_qa` (SQLite Q&A DB the LLM can read and write).
- **Evaluator:** After each reply, an LLM evaluates it; if unacceptable, we rerun with feedback once.
- **Pushover:** Optional push notifications to My phone (e.g. when someone leaves their email or asks something we couldn’t answer).

---

### Pushover setup (optional)

Pushover sends push notifications to your phone. To set it up:

1. Visit https://pushover.net/ and sign up; create an application/API token (e.g. name it "Agents").
2. Add to your `.env` file:
   - `PUSHOVER_USER=` _(key on your Pushover home screen, often starts with `u`)_
   - `PUSHOVER_TOKEN=` _(token from your new application, often starts with `a`)_
3. Save `.env` and run `load_dotenv(override=True)` after saving.
4. Install the Pushover app on your phone/tablet so you receive the notifications.


In [ ]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
import sqlite3
from pydantic import BaseModel
from pypdf import PdfReader
import gradio as gr

In [ ]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [ ]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [ ]:
# SQLite Q&A database for common questions the LLM can read and write

QA_DB_PATH = "qa.db"

def _init_qa_db():
    conn = sqlite3.connect(QA_DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS qa (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            question TEXT NOT NULL,
            answer TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

_init_qa_db()

def query_qa(question=None):
    """Look up Q&A pairs. If question is given, return matching rows; otherwise return all (or recent) pairs."""
    print("Q&A query called")
    conn = sqlite3.connect(QA_DB_PATH)
    conn.row_factory = sqlite3.Row
    if question and question.strip():
        cur = conn.execute(
            "SELECT question, answer FROM qa WHERE question LIKE ? OR answer LIKE ? ORDER BY id DESC LIMIT 10",
            (f"%{question.strip()}%", f"%{question.strip()}%"),
        )
    else:
        cur = conn.execute("SELECT question, answer FROM qa ORDER BY id DESC LIMIT 20")
    rows = [dict(r) for r in cur.fetchall()]
    conn.close()
    return {"count": len(rows), "pairs": rows}

def upsert_qa(question: str, answer: str):
    """Add or update a Q&A pair. Use when the user or you establish a new common Q&A to store for future use."""
    print("Upsert Q&A called")
    conn = sqlite3.connect(QA_DB_PATH)
    conn.execute("INSERT INTO qa (question, answer) VALUES (?, ?)", (question.strip(), answer.strip()))
    conn.commit()
    conn.close()
    return {"recorded": "ok"}

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
# Tool definitions for SQL Q&A (LLM can read and write common Q&A)

query_qa_json = {
    "name": "query_qa",
    "description": "Look up stored Q&A pairs. Use when answering common or recurring questions. Pass a search string to find matching questions/answers, or omit to get recent pairs.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Optional search string to filter Q&A pairs (e.g. 'availability', 'contact'). Omit or empty to list recent pairs.",
            }
        },
        "required": [],
        "additionalProperties": False,
    },
}

upsert_qa_json = {
    "name": "upsert_qa",
    "description": "Store a new Q&A pair for future use. Use when the user asks something you answer and you want to remember it for next time (e.g. preferred contact method, availability).",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question or topic."},
            "answer": {"type": "string", "description": "The answer to store."},
        },
        "required": ["question", "answer"],
        "additionalProperties": False,
    },
}

In [ ]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
    {"type": "function", "function": query_qa_json},
    {"type": "function", "function": upsert_qa_json},
]

In [ ]:
tools

In [ ]:
# Dispatch tool calls via an explicit mapping (no IF chain, no globals)

TOOL_MAP = {
    "record_user_details": record_user_details,
    "record_unknown_question": record_unknown_question,
    "query_qa": query_qa,
    "upsert_qa": upsert_qa,
}


def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = TOOL_MAP.get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [ ]:
# Load context first (required by system_prompt and evaluator below)
reader = PdfReader("linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Chukwunonso Ikeji"
other_name = "William"
alias = "Codypharm"

In [ ]:
system_prompt = f"You are acting as {name} whose other name is {other_name} and has an alias {alias}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. \
Use query_qa to look up stored Q&A for common questions; use upsert_qa to store new Q&A when the user shares something worth remembering (e.g. contact preference, availability). "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [ ]:
# Evaluator: decide if the agent's reply is acceptable 

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [ ]:
evaluator_system_prompt = (
    f"You are an evaluator that decides whether the Agent's response to a user question is acceptable. "
    f"The Agent is playing the role of {name} whose other name is {other_name} and has an alias {alias}. "
    f"Check that the response is accurate (no hallucination), on-topic, professional, and helpful. "
    f"Reply with is_acceptable (true/false) and brief feedback."
)
evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn profile (excerpt):\n{linkedin}\n\n"

def evaluator_user_prompt(reply: str, message: str, history: list) -> str:
    conv = "\n".join(
        f"{h.get('role', 'user')}: {h.get('content', '')[:200]}"
        for h in history
    ) if history else "(no prior messages)"
    return f"Conversation so far:\n{conv}\n\nUser's latest message: {message}\n\nAgent's latest response: {reply}\n\nEvaluate: is this response acceptable and in character?"

In [ ]:
def evaluate(reply: str, message: str, history: list) -> Evaluation:
    """Run the evaluator LLM on the agent's reply."""
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
    ]
    response = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=messages,
        response_format=Evaluation,
    )
    return response.choices[0].message.parsed

def rerun(reply: str, message: str, history: list, feedback: str) -> str:
    """Regenerate a reply with evaluator feedback (critique-and-refine)."""
    updated_system = (
        system_prompt
        + "\n\n[Previous reply was rejected by quality check.]\n"
        + f"Your attempted answer: {reply[:500]}...\n"
        + f"Feedback: {feedback}\n"
        + "Please reply again, addressing the feedback."
    )
    messages = [{"role": "system", "content": updated_system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            msg = response.choices[0].message
            tool_calls = msg.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(msg)
            messages.extend(results)
        else:
            done = True
    reply = response.choices[0].message.content
    evaluation = evaluate(reply, message, history)
    if not evaluation.is_acceptable:
        reply = rerun(reply, message, history, evaluation.feedback)
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()